# SQL and Data Manipulation for ML
## JOINs, Window Functions, CTEs, and Feature Engineering

This notebook demonstrates SQL concepts essential for ML pipelines:
- JOINs (INNER, LEFT, RIGHT)
- Window functions (ROW_NUMBER, RANK, LAG, LEAD)
- Common Table Expressions (CTEs)
- Feature engineering in SQL
- Temporal data handling to prevent leakage
- Data quality checks
- Creating training datasets from relational data

📺 **Video Lecture:** [https://youtu.be/HLPBLmkYABE](https://youtu.be/HLPBLmkYABE)

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

## 1. Create Sample Datasets

Build a realistic e-commerce scenario with users, transactions, and products.

In [3]:
# Create users table
users_df = pd.DataFrame({
    'user_id': range(1, 101),
    'signup_date': [datetime(2023, 1, 1) + timedelta(days=int(x)) for x in np.random.uniform(0, 365, 100)],
    'country': np.random.choice(['US', 'UK', 'CA', 'DE'], 100),
    'age': np.random.normal(35, 15, 100).astype(int)
})
users_df.to_sql('users', conn, index=False, if_exists='replace')

# Create transactions table
transactions_data = []
for user_id in range(1, 101):
    n_transactions = np.random.poisson(5)  # Poisson-distributed
    for _ in range(n_transactions):
        date = datetime(2024, 1, 1) + timedelta(days=int(np.random.uniform(0, 90)))
        amount = np.random.exponential(50) + 10  # Positive amounts
        transactions_data.append({
            'transaction_id': len(transactions_data) + 1,
            'user_id': user_id,
            'transaction_date': date,
            'amount': amount,
            'product_id': np.random.randint(1, 51)
        })

transactions_df = pd.DataFrame(transactions_data)
transactions_df.to_sql('transactions', conn, index=False, if_exists='replace')

# Create products table
products_df = pd.DataFrame({
    'product_id': range(1, 51),
    'category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home'], 50),
    'price': np.random.exponential(30) + 5
})
products_df.to_sql('products', conn, index=False, if_exists='replace')

print(f"Users: {len(users_df)}")
print(f"Transactions: {len(transactions_df)}")
print(f"Products: {len(products_df)}")
print(f"\nUsers preview:")
print(users_df.head(3))

Users: 100
Transactions: 523
Products: 50

Users preview:
   user_id signup_date country  age
0        1  2023-05-17      CA   32
1        2  2023-12-14      DE   39
2        3  2023-09-25      CA   34


## 2. JOIN Operations

Combine data from multiple tables using different JOIN types.

In [4]:
# INNER JOIN: Only users with transactions
inner_join_query = """
SELECT 
    u.user_id,
    u.country,
    COUNT(*) as num_transactions,
    SUM(t.amount) as total_spending
FROM users u
INNER JOIN transactions t ON u.user_id = t.user_id
GROUP BY u.user_id, u.country
ORDER BY total_spending DESC
LIMIT 10
"""

inner_result = pd.read_sql_query(inner_join_query, conn)
print("INNER JOIN: Users with transactions")
print(inner_result)
print(f"\nRows returned: {len(inner_result)} (users with at least 1 transaction)")

# LEFT JOIN: All users, even those without transactions
left_join_query = """
SELECT 
    u.user_id,
    u.country,
    COUNT(DISTINCT t.transaction_id) as num_transactions,
    COALESCE(SUM(t.amount), 0) as total_spending
FROM users u
LEFT JOIN transactions t ON u.user_id = t.user_id
GROUP BY u.user_id, u.country
ORDER BY total_spending DESC
LIMIT 10
"""

left_result = pd.read_sql_query(left_join_query, conn)
print("\nLEFT JOIN: All users (including those with no transactions)")
print(left_result)

# Show difference
print(f"\nINNER JOIN rows: {len(inner_result)}")
print(f"LEFT JOIN rows: {len(left_result)}")
print(f"Difference: {len(left_result) - len(inner_result)} users have no transactions")

INNER JOIN: Users with transactions
   user_id country  num_transactions  total_spending
0       38      UK                10      944.623663
1        8      DE                 9      793.478620
2       35      DE                10      743.247714
3       23      UK                 9      708.099641
4       85      UK                 9      647.696616
5       36      DE                 8      629.666273
6       22      CA                 7      589.018853
7       70      CA                 8      574.275069
8       50      CA                 8      558.840770
9       75      US                 6      548.542412

Rows returned: 10 (users with at least 1 transaction)

LEFT JOIN: All users (including those with no transactions)
   user_id country  num_transactions  total_spending
0       38      UK                10      944.623663
1        8      DE                 9      793.478620
2       35      DE                10      743.247714
3       23      UK                 9      708.099641


## 3. Window Functions

Compute running statistics and rankings within groups.

In [5]:
# Window function: rank users by spending per country
window_query = """
SELECT 
    u.user_id,
    u.country,
    SUM(t.amount) as total_spending,
    ROW_NUMBER() OVER (PARTITION BY u.country ORDER BY SUM(t.amount) DESC) as rank_in_country,
    RANK() OVER (PARTITION BY u.country ORDER BY SUM(t.amount) DESC) as rank_with_ties,
    COUNT(*) OVER (PARTITION BY u.country) as users_in_country
FROM users u
LEFT JOIN transactions t ON u.user_id = t.user_id
GROUP BY u.user_id, u.country
ORDER BY u.country, total_spending DESC
LIMIT 15
"""

window_result = pd.read_sql_query(window_query, conn)
print("Window Functions: Ranking by Country")
print(window_result)

# LAG and LEAD for time-series
lag_lead_query = """
WITH user_spending_timeline AS (
    SELECT 
        u.user_id,
        DATE(t.transaction_date) as date,
        SUM(t.amount) as daily_spending
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    WHERE u.user_id <= 5
    GROUP BY u.user_id, DATE(t.transaction_date)
)
SELECT 
    user_id,
    date,
    daily_spending,
    LAG(daily_spending) OVER (PARTITION BY user_id ORDER BY date) as prev_day_spending,
    LEAD(daily_spending) OVER (PARTITION BY user_id ORDER BY date) as next_day_spending,
    (daily_spending - LAG(daily_spending) OVER (PARTITION BY user_id ORDER BY date)) as spending_change
FROM user_spending_timeline
LIMIT 10
"""

lag_result = pd.read_sql_query(lag_lead_query, conn)
print("\nLAG and LEAD: Time-series Analysis")
print(lag_result)

Window Functions: Ranking by Country
    user_id country  total_spending  rank_in_country  rank_with_ties  \
0        22      CA      589.018853                1               1   
1        70      CA      574.275069                2               2   
2        50      CA      558.840770                3               3   
3        42      CA      512.090638                4               4   
4        15      CA      438.656344                5               5   
5        65      CA      430.110730                6               6   
6        92      CA      419.873513                7               7   
7        69      CA      393.996610                8               8   
8        57      CA      330.640525                9               9   
9        12      CA      324.094487               10              10   
10       45      CA      320.382674               11              11   
11       14      CA      254.490582               12              12   
12       11      CA      24

## 4. Common Table Expressions (CTEs)

Use WITH clauses to create readable, modular queries.

In [6]:
# Multi-level CTEs for feature engineering
cte_query = """
WITH user_transactions AS (
    -- Step 1: Aggregate transactions per user
    SELECT 
        u.user_id,
        u.country,
        u.age,
        COUNT(DISTINCT t.transaction_id) as num_transactions,
        SUM(t.amount) as total_spending,
        AVG(t.amount) as avg_transaction_value,
        MIN(t.transaction_date) as first_purchase_date,
        MAX(t.transaction_date) as last_purchase_date
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    GROUP BY u.user_id, u.country, u.age
),
user_product_diversity AS (
    -- Step 2: Count unique products purchased
    SELECT 
        u.user_id,
        COUNT(DISTINCT t.product_id) as unique_products,
        COUNT(DISTINCT p.category) as unique_categories
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    LEFT JOIN products p ON t.product_id = p.product_id
    GROUP BY u.user_id
)
SELECT 
    ut.user_id,
    ut.country,
    ut.age,
    ut.num_transactions,
    ut.total_spending,
    ut.avg_transaction_value,
    upd.unique_products,
    upd.unique_categories,
    CASE 
        WHEN ut.num_transactions = 0 THEN 'No Purchase'
        WHEN ut.total_spending < 100 THEN 'Low Value'
        WHEN ut.total_spending < 500 THEN 'Medium Value'
        ELSE 'High Value'
    END as customer_segment
FROM user_transactions ut
LEFT JOIN user_product_diversity upd ON ut.user_id = upd.user_id
ORDER BY ut.total_spending DESC
LIMIT 10
"""

cte_result = pd.read_sql_query(cte_query, conn)
print("CTEs: Multi-level Feature Engineering")
print(cte_result)

CTEs: Multi-level Feature Engineering
   user_id country  age  num_transactions  total_spending  \
0       38      UK   40                10      944.623663   
1        8      DE   21                 9      793.478620   
2       35      DE   42                10      743.247714   
3       23      UK   23                 9      708.099641   
4       85      UK   35                 9      647.696616   
5       36      DE   39                 8      629.666273   
6       22      CA   58                 7      589.018853   
7       70      CA   42                 8      574.275069   
8       50      CA   22                 8      558.840770   
9       75      US   22                 6      548.542412   

   avg_transaction_value  unique_products  unique_categories customer_segment  
0              94.462366               10                  4       High Value  
1              88.164291                8                  4       High Value  
2              74.324771                8         

## 5. Feature Engineering in SQL

In [7]:
# Feature engineering for predictive modeling
feature_query = """
WITH user_stats AS (
    SELECT 
        u.user_id,
        u.country,
        u.age,
        (julianday('2024-03-31') - julianday(u.signup_date)) as days_as_customer,
        COUNT(DISTINCT t.transaction_id) as total_purchases,
        SUM(t.amount) as lifetime_value,
        AVG(t.amount) as avg_purchase_value,
        MAX(t.amount) as max_purchase,
        MIN(t.amount) as min_purchase,
        CAST(MAX(t.transaction_date) AS DATE) as last_purchase
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    GROUP BY u.user_id
)
SELECT 
    user_id,
    country,
    age,
    days_as_customer,
    total_purchases,
    lifetime_value,
    avg_purchase_value,
    -- Derived features
    ROUND(lifetime_value / NULLIF(days_as_customer, 0), 2) as daily_spending_rate,
    ROUND(total_purchases / NULLIF(days_as_customer, 0) * 30, 2) as monthly_purchase_frequency,
    CASE WHEN total_purchases = 0 THEN 1 ELSE 0 END as is_inactive,
    CASE WHEN (julianday('2024-03-31') - julianday(last_purchase)) > 30 THEN 1 ELSE 0 END as churned_30days,
    CASE 
        WHEN total_purchases < 2 THEN 'New'
        WHEN total_purchases < 5 THEN 'Regular'
        ELSE 'Loyal'
    END as customer_type
FROM user_stats
WHERE total_purchases > 0
ORDER BY lifetime_value DESC
LIMIT 10
"""

features_df = pd.read_sql_query(feature_query, conn)
print("Engineered Features for ML:")
print(features_df)

Engineered Features for ML:
   user_id country  age  days_as_customer  total_purchases  lifetime_value  \
0       38      UK   40             420.0               10      944.623663   
1        8      DE   21             139.0                9      793.478620   
2       35      DE   42             103.0               10      743.247714   
3       23      UK   23             349.0                9      708.099641   
4       85      UK   35             342.0                9      647.696616   
5       36      DE   39             160.0                8      629.666273   
6       22      CA   58             405.0                7      589.018853   
7       70      CA   42              95.0                8      574.275069   
8       50      CA   22             388.0                8      558.840770   
9       75      US   22             197.0                6      548.542412   

   avg_purchase_value  daily_spending_rate  monthly_purchase_frequency  \
0           94.462366                 2

## 6. Temporal Data Handling (Leakage Prevention)

In [8]:
# Time-based train/test split with NO data leakage
leakage_prevention_query = """
WITH raw_data AS (
    SELECT 
        u.user_id,
        t.transaction_date,
        t.amount,
        ROW_NUMBER() OVER (PARTITION BY u.user_id ORDER BY t.transaction_date) as purchase_order
    FROM users u
    INNER JOIN transactions t ON u.user_id = t.user_id
),
features_at_observation_point AS (
    -- Features computed BEFORE the observation date (2024-02-15)
    SELECT 
        u.user_id,
        DATE('2024-02-15') as observation_date,
        COUNT(DISTINCT CASE WHEN t.transaction_date < '2024-02-15' THEN t.transaction_id END) as history_purchases,
        SUM(CASE WHEN t.transaction_date < '2024-02-15' THEN t.amount ELSE 0 END) as history_spending,
        COUNT(DISTINCT CASE WHEN t.transaction_date >= '2024-02-15' AND t.transaction_date < DATE('2024-02-15', '+30 days') THEN t.transaction_id END) as future_purchases_30d
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    GROUP BY u.user_id
)
SELECT 
    user_id,
    observation_date,
    history_purchases,
    history_spending,
    CASE WHEN future_purchases_30d > 0 THEN 1 ELSE 0 END as active_30d_label
FROM features_at_observation_point
WHERE history_purchases > 0
ORDER BY history_spending DESC
LIMIT 10
"""

train_set = pd.read_sql_query(leakage_prevention_query, conn)
print("Training Data with Leakage Prevention:")
print(train_set)
print("\nNote: Features computed BEFORE observation date")
print("      Target computed AFTER observation date")
print("      No data leakage!")

Training Data with Leakage Prevention:
   user_id observation_date  history_purchases  history_spending  \
0       38       2024-02-15                  6        664.058679   
1       70       2024-02-15                  6        412.964634   
2       19       2024-02-15                  5        375.466376   
3       69       2024-02-15                  5        375.173818   
4       34       2024-02-15                  3        352.257011   
5       50       2024-02-15                  5        340.214965   
6       79       2024-02-15                  4        340.014826   
7       15       2024-02-15                  4        332.624830   
8        2       2024-02-15                  5        325.854263   
9       57       2024-02-15                  3        319.763896   

   active_30d_label  
0                 1  
1                 1  
2                 0  
3                 0  
4                 1  
5                 1  
6                 0  
7                 1  
8             

## 7. Data Quality Checks

In [9]:
# Data quality validation
quality_query = """
SELECT 
    'Users' as table_name,
    COUNT(*) as total_rows,
    COUNT(DISTINCT user_id) as distinct_ids,
    COUNT(CASE WHEN age IS NULL THEN 1 END) as null_age,
    COUNT(CASE WHEN age < 0 OR age > 150 THEN 1 END) as invalid_age,
    CAST(COUNT(CASE WHEN age < 0 OR age > 150 THEN 1 END) AS FLOAT) / COUNT(*) as invalid_pct
FROM users

UNION ALL

SELECT 
    'Transactions' as table_name,
    COUNT(*) as total_rows,
    COUNT(DISTINCT transaction_id) as distinct_ids,
    COUNT(CASE WHEN amount IS NULL THEN 1 END) as null_amount,
    COUNT(CASE WHEN amount <= 0 THEN 1 END) as invalid_amount,
    CAST(COUNT(CASE WHEN amount <= 0 THEN 1 END) AS FLOAT) / COUNT(*) as invalid_pct
FROM transactions
"""

quality_results = pd.read_sql_query(quality_query, conn)
print("Data Quality Report:")
print(quality_results)

# Check for foreign key violations
fk_check = """
SELECT 
    COUNT(*) as transactions_with_invalid_user,
    COUNT(CASE WHEN p.product_id IS NULL THEN 1 END) as transactions_with_invalid_product
FROM transactions t
LEFT JOIN users u ON t.user_id = u.user_id
LEFT JOIN products p ON t.product_id = p.product_id
WHERE u.user_id IS NULL OR p.product_id IS NULL
"""

fk_results = pd.read_sql_query(fk_check, conn)
print("\nForeign Key Check:")
print(fk_results)

Data Quality Report:
     table_name  total_rows  distinct_ids  null_age  invalid_age  invalid_pct
0         Users         100           100         0            0          0.0
1  Transactions         523           523         0            0          0.0

Foreign Key Check:
   transactions_with_invalid_user  transactions_with_invalid_product
0                               0                                  0


## 8. Summary: SQL for ML Pipelines

In [10]:
# Close connection
conn.close()

print("\n" + "="*90)
print("SQL BEST PRACTICES FOR ML PIPELINES")
print("="*90)
print("\n1. JOINs:")
print("   - INNER JOIN: Only matched records (excludes nulls)")
print("   - LEFT JOIN: Keep left table, add right (handle nulls!)")
print("   - Ensure join keys are indexed for performance")
print("\n2. Window Functions:")
print("   - ROW_NUMBER(): Strict ordering, useful for ranking")
print("   - RANK(): Handles ties")
print("   - LAG/LEAD: Access previous/next rows for time-series")
print("\n3. CTEs (WITH clauses):")
print("   - Break complex queries into readable steps")
print("   - Reusable intermediate tables")
print("   - Easier to debug")
print("\n4. Feature Engineering:")
print("   - Aggregations: COUNT, SUM, AVG, MIN, MAX")
print("   - Derived features: Ratios, flags, categorizations")
print("   - CASE statements for bucketing")
print("\n5. Temporal/Leakage Prevention:")
print("   - Feature cutoff date: features BEFORE this date")
print("   - Target: computed AFTER cutoff date")
print("   - Never use future information in features!")
print("\n6. Data Quality:")
print("   - Check nulls, duplicates, out-of-range values")
print("   - Validate foreign keys")
print("   - Document data dictionary")
print("="*90)


SQL BEST PRACTICES FOR ML PIPELINES

1. JOINs:
   - INNER JOIN: Only matched records (excludes nulls)
   - LEFT JOIN: Keep left table, add right (handle nulls!)
   - Ensure join keys are indexed for performance

2. Window Functions:
   - ROW_NUMBER(): Strict ordering, useful for ranking
   - RANK(): Handles ties
   - LAG/LEAD: Access previous/next rows for time-series

3. CTEs (WITH clauses):
   - Break complex queries into readable steps
   - Reusable intermediate tables
   - Easier to debug

4. Feature Engineering:
   - Aggregations: COUNT, SUM, AVG, MIN, MAX
   - Derived features: Ratios, flags, categorizations
   - CASE statements for bucketing

5. Temporal/Leakage Prevention:
   - Feature cutoff date: features BEFORE this date
   - Target: computed AFTER cutoff date
   - Never use future information in features!

6. Data Quality:
   - Check nulls, duplicates, out-of-range values
   - Validate foreign keys
   - Document data dictionary


---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>